In [27]:
from pathlib import Path
import random

import numpy as np
import pandas as pd
import matplotlib.pyplot as plt

import torch
import torch.nn as nn
import torch.nn.functional as F
from torch.utils.data import Dataset, DataLoader

In [28]:
def setSeed(seed=42):
    """
    Set random seeds for reproducibility.
    """
    random.seed(seed)
    np.random.seed(seed)
    torch.manual_seed(seed)

    if torch.cuda.is_available():
        torch.cuda.manual_seed(seed)
        torch.cuda.manual_seed_all(seed)


setSeed(42)

DEVICE = torch.device("cuda" if torch.cuda.is_available() else "cpu")
print("Using device:", DEVICE)

Using device: cpu


In [29]:
from pathlib import Path

IN_COLAB = False
try:
    import google.colab  # type: ignore
    IN_COLAB = True
except ImportError:
    IN_COLAB = False

if IN_COLAB:
    from google.colab import drive
    drive.mount("/content/drive")
    ROOT = Path("/content/drive/MyDrive/ASR_Project/ASR_finalproject_AllisonWivagg_YeritmaryR")
else:
    currentDirectory = Path.cwd()

    if (currentDirectory / "data").exists():
        ROOT = currentDirectory
    elif (currentDirectory.parent / "data").exists():
        ROOT = currentDirectory.parent
    elif (currentDirectory.parent.parent / "data").exists():
        ROOT = currentDirectory.parent.parent
    else:
        raise FileNotFoundError("Could not find local project root containing 'data/'.")

dataDirectory = ROOT / "data"

b006ImageDirectory = dataDirectory / "b006" / "images"
b006MaskDirectory = dataDirectory / "b006" / "masks"

b039ImageDirectory = dataDirectory / "b039" / "images"
b039MaskDirectory = dataDirectory / "b039" / "masks"

combinedImageDirectory = dataDirectory / "combined" / "images"
combinedMaskDirectory = dataDirectory / "combined" / "masks"

print("ROOT:", ROOT)
print("b039ImageDirectory exists:", b039ImageDirectory.exists())
print("b039MaskDirectory exists:", b039MaskDirectory.exists())

Drive already mounted at /content/drive; to attempt to forcibly remount, call drive.mount("/content/drive", force_remount=True).
ROOT: /content/drive/MyDrive/ASR_Project/ASR_finalproject_AllisonWivagg_YeritmaryR
b039ImageDirectory exists: True
b039MaskDirectory exists: True


In [30]:
# Using models for training and evaluation 

class DoubleConv(nn.Module):
    """
    Two convolution blocks with batch normalization, ReLU, and dropout.
    """

    def __init__(self, inputChannels, outputChannels, dropoutRate=0.3):
        super().__init__()
        self.block = nn.Sequential(
            nn.Conv2d(inputChannels, outputChannels, kernel_size=3, padding=1),
            nn.BatchNorm2d(outputChannels),
            nn.ReLU(inplace=True),
            nn.Dropout2d(p=dropoutRate),

            nn.Conv2d(outputChannels, outputChannels, kernel_size=3, padding=1),
            nn.BatchNorm2d(outputChannels),
            nn.ReLU(inplace=True),
            nn.Dropout2d(p=dropoutRate),
        )

    def forward(self, x):
        return self.block(x)


class UNet(nn.Module):
    """
    U-Net for binary nuclei segmentation.
    """

    def __init__(self, inputChannels=1, outputChannels=1, featureSizes=(32, 64, 128, 256), dropoutRate=0.3):
        super().__init__()

        self.downBlocks = nn.ModuleList()
        self.poolLayers = nn.ModuleList()

        currentChannels = inputChannels
        for featureCount in featureSizes:
            self.downBlocks.append(DoubleConv(currentChannels, featureCount, dropoutRate=dropoutRate))
            self.poolLayers.append(nn.MaxPool2d(kernel_size=2, stride=2))
            currentChannels = featureCount

        self.bottleneck = DoubleConv(featureSizes[-1], featureSizes[-1] * 2, dropoutRate=dropoutRate)

        self.upSamples = nn.ModuleList()
        self.upBlocks = nn.ModuleList()

        currentChannels = featureSizes[-1] * 2
        for featureCount in featureSizes[::-1]:
            self.upSamples.append(nn.ConvTranspose2d(currentChannels, featureCount, kernel_size=2, stride=2))
            self.upBlocks.append(DoubleConv(featureCount * 2, featureCount, dropoutRate=dropoutRate))
            currentChannels = featureCount

        self.outputLayer = nn.Conv2d(featureSizes[0], outputChannels, kernel_size=1)

    def forward(self, x):
        skipConnections = []

        for downBlock, poolLayer in zip(self.downBlocks, self.poolLayers):
            x = downBlock(x)
            skipConnections.append(x)
            x = poolLayer(x)

        x = self.bottleneck(x)
        skipConnections = skipConnections[::-1]

        for blockIndex in range(len(self.upSamples)):
            x = self.upSamples[blockIndex](x)
            skipTensor = skipConnections[blockIndex]

            if x.shape[2:] != skipTensor.shape[2:]:
                x = F.interpolate(x, size=skipTensor.shape[2:], mode="bilinear", align_corners=False)

            x = torch.cat([skipTensor, x], dim=1)
            x = self.upBlocks[blockIndex](x)

        return self.outputLayer(x)

In [31]:
#Calculating the segmentation metrics for evaluation of the model performance. and dice loss and IoU loss for training the model.

class SegLoss(nn.Module):
    """
    BCE-with-logits loss plus Dice loss for binary segmentation.
    """

    def __init__(self, smooth=1e-6):
        super().__init__()
        self.binaryCrossEntropy = nn.BCEWithLogitsLoss()
        self.smooth = smooth

    def forward(self, predictionLogits, trueMasks):
        binaryLoss = self.binaryCrossEntropy(predictionLogits, trueMasks)

        predictionProbabilities = torch.sigmoid(predictionLogits)
        predictionProbabilities = predictionProbabilities.view(predictionProbabilities.size(0), -1)
        trueMasks = trueMasks.view(trueMasks.size(0), -1)

        intersection = (predictionProbabilities * trueMasks).sum(dim=1)
        diceScore = (2.0 * intersection + self.smooth) / (
            predictionProbabilities.sum(dim=1) + trueMasks.sum(dim=1) + self.smooth
        )
        diceLoss = 1.0 - diceScore.mean()

        return binaryLoss + diceLoss


def maskScores(predictionLogits, trueMasks, threshold=0.5, smooth=1e-6):
    """
    Compute batch Dice and IoU from logits and true masks.
    """
    predictionProbabilities = torch.sigmoid(predictionLogits)
    predictedMasks = (predictionProbabilities > threshold).float()

    predictedMasks = predictedMasks.view(predictedMasks.size(0), -1)
    trueMasks = trueMasks.view(trueMasks.size(0), -1)

    intersection = (predictedMasks * trueMasks).sum(dim=1)
    union = predictedMasks.sum(dim=1) + trueMasks.sum(dim=1) - intersection

    diceScore = (
        (2.0 * intersection + smooth) /
        (predictedMasks.sum(dim=1) + trueMasks.sum(dim=1) + smooth)
    ).mean().item()

    iouScore = (
        (intersection + smooth) /
        (union + smooth)
    ).mean().item()

    return {
        "dice": diceScore,
        "iou": iouScore
    }

In [32]:
# Testing the UNet architecture with a dummy input to verify the output shape.
testNetwork = UNet(inputChannels=1, outputChannels=1, dropoutRate=0.3).to(DEVICE)

dummyInput = torch.randn(1, 1, 256, 256).to(DEVICE)
dummyOutput = testNetwork(dummyInput)

print("input shape:", dummyInput.shape)
print("output shape:", dummyOutput.shape)

input shape: torch.Size([1, 1, 256, 256])
output shape: torch.Size([1, 1, 256, 256])


In [33]:
#Evaluating the network with dummy data to verify the loss and metric calculations.

def evaluateNetwork(network, dataLoader, lossFunction, device):
    """
    Evaluate a segmentation model on one dataloader.

    Returns:
        Dictionary with average loss, Dice, and IoU.
    """
    network.eval()

    totalLoss = 0.0
    totalDice = 0.0
    totalIoU = 0.0
    totalBatches = 0

    with torch.no_grad():
        for batch in dataLoader:
            images = batch["image"].to(device)
            trueMasks = batch["mask"].to(device)

            predictionLogits = network(images)
            loss = lossFunction(predictionLogits, trueMasks)
            scoreTable = maskScores(predictionLogits, trueMasks)

            totalLoss += loss.item()
            totalDice += scoreTable["dice"]
            totalIoU += scoreTable["iou"]
            totalBatches += 1

    if totalBatches == 0:
        return {"loss": np.nan, "dice": np.nan, "iou": np.nan}

    return {
        "loss": totalLoss / totalBatches,
        "dice": totalDice / totalBatches,
        "iou": totalIoU / totalBatches
    }

In [34]:
# Training the network with the training dataloader and tracking validation performance after each epoch. 
# Saving the best model based on validation loss.
def trainNetwork(
    network,
    trainingLoader,
    validationLoader,
    device,
    numEpochs=5,
    learningRate=1e-3,
    savingPath=None
):
    """
    Train a segmentation model and track validation performance.

    Returns:
        trained network, history table
    """
    optimizer = torch.optim.Adam(network.parameters(), lr=learningRate)
    lossFunction = SegLoss()

    bestValidationLoss = float("inf")
    historyRows = []

    for epochIndex in range(numEpochs):
        network.train()

        totalTrainingLoss = 0.0
        totalTrainingBatches = 0

        for batch in trainingLoader:
            images = batch["image"].to(device)
            trueMasks = batch["mask"].to(device)

            optimizer.zero_grad()
            predictionLogits = network(images)
            loss = lossFunction(predictionLogits, trueMasks)
            loss.backward()
            optimizer.step()

            totalTrainingLoss += loss.item()
            totalTrainingBatches += 1

        averageTrainingLoss = totalTrainingLoss / max(totalTrainingBatches, 1)

        validationScores = evaluateNetwork(
            network=network,
            dataLoader=validationLoader,
            lossFunction=lossFunction,
            device=device
        )

        historyRows.append({
            "epoch": epochIndex + 1,
            "trainLoss": averageTrainingLoss,
            "validationLoss": validationScores["loss"],
            "validationDice": validationScores["dice"],
            "validationIoU": validationScores["iou"]
        })

        print(
            f"Epoch {epochIndex + 1}/{numEpochs} | "
            f"Train Loss: {averageTrainingLoss:.4f} | "
            f"Validation Loss: {validationScores['loss']:.4f} | "
            f"Validation Dice: {validationScores['dice']:.4f} | "
            f"Validation IoU: {validationScores['iou']:.4f}"
        )

        if validationScores["loss"] < bestValidationLoss:
            bestValidationLoss = validationScores["loss"]

            if savingPath is not None:
                savingPath = Path(savingPath)
                savingPath.parent.mkdir(parents=True, exist_ok=True)
                torch.save(network.state_dict(), savingPath)

    if savingPath is not None and Path(savingPath).exists():
        network.load_state_dict(torch.load(savingPath, map_location=device))

    historyTable = pd.DataFrame(historyRows)
    return network, historyTable

In [35]:
#Indexing the dataset and loading it into the network for training and evaluation on the small dataset

class IndexedSet(Dataset):
    """
    Thin wrapper that keeps only selected indices from a dataset.
    """

    def __init__(self, dataset, selectedIndices):
        self.dataset = dataset
        self.selectedIndices = list(selectedIndices)

    def __len__(self):
        return len(self.selectedIndices)

    def __getitem__(self, index):
        datasetIndex = self.selectedIndices[index]
        item = self.dataset[datasetIndex]
        item["dataset_index"] = torch.tensor(datasetIndex, dtype=torch.long)
        return item


def makeLoader(dataset, batchSize=4, shuffle=False, numWorkers=0):
    """
    Create a PyTorch DataLoader.
    """
    return DataLoader(
        dataset,
        batch_size=batchSize,
        shuffle=shuffle,
        num_workers=numWorkers
    )

In [36]:
from pathlib import Path
import numpy as np
import pandas as pd
import tifffile as tiff
from PIL import Image
import torch
from torch.utils.data import Dataset
from sklearn.model_selection import train_test_split

def fileStem(path):
    return Path(path).stem

def listImages(imageDirectory):
    extensions = ["*.png", "*.tif", "*.tiff", "*.jpg", "*.jpeg"]
    imageFiles = []
    for extension in extensions:
        imageFiles.extend(Path(imageDirectory).glob(extension))
    return sorted(imageFiles)

def pairFiles(imageDirectory, maskDirectory, sourceName):
    imageFiles = listImages(Path(imageDirectory))
    maskFiles = listImages(Path(maskDirectory))

    imageMap = {fileStem(path): path for path in imageFiles}
    pairedRows = []

    for maskPath in maskFiles:
        imageId = fileStem(maskPath)
        if imageId in imageMap:
            pairedRows.append({
                "image_id": imageId,
                "image_path": str(imageMap[imageId]),
                "mask_path": str(maskPath),
                "source": sourceName
            })

    pairedTable = pd.DataFrame(pairedRows)

    if len(pairedTable) == 0:
        raise ValueError(f"No matched image-mask pairs found for source {sourceName}")

    pairedTable = pairedTable.sort_values(["source", "image_id"]).reset_index(drop=True)
    return pairedTable

def pairB006(imageDirectory, maskDirectory, sourceName="BBBC006", channelTag="_w1"):
    imageFiles = listImages(Path(imageDirectory))
    maskFiles = listImages(Path(maskDirectory))

    pairedRows = []

    for maskPath in maskFiles:
        maskId = fileStem(maskPath)

        candidateImages = []
        for imagePath in imageFiles:
            imageId = fileStem(imagePath)
            if imageId.startswith(maskId + channelTag):
                candidateImages.append(imagePath)

        if len(candidateImages) == 1:
            pairedRows.append({
                "image_id": maskId,
                "image_path": str(candidateImages[0]),
                "mask_path": str(maskPath),
                "source": sourceName
            })

    pairedTable = pd.DataFrame(pairedRows)

    if len(pairedTable) == 0:
        raise ValueError(f"No matched image-mask pairs found for source {sourceName}")

    pairedTable = pairedTable.sort_values(["source", "image_id"]).reset_index(drop=True)
    return pairedTable

def splitData(dataTable, seed=42, trainSize=0.70, validationSize=0.15, testSize=0.15):
    totalSize = trainSize + validationSize + testSize
    if not np.isclose(totalSize, 1.0):
        raise ValueError("trainSize + validationSize + testSize must sum to 1.0")

    trainTable, tempTable = train_test_split(
        dataTable,
        test_size=(1.0 - trainSize),
        random_state=seed,
        shuffle=True
    )

    relativeTestSize = testSize / (validationSize + testSize)

    validationTable, testTable = train_test_split(
        tempTable,
        test_size=relativeTestSize,
        random_state=seed,
        shuffle=True
    )

    trainTable = trainTable.reset_index(drop=True).copy()
    validationTable = validationTable.reset_index(drop=True).copy()
    testTable = testTable.reset_index(drop=True).copy()

    trainTable["split"] = "train"
    validationTable["split"] = "validation"
    testTable["split"] = "test"

    return trainTable, validationTable, testTable

In [37]:
b039Table = pairFiles(b039ImageDirectory, b039MaskDirectory, "BBBC039")
trainTable, validationTable, testTable = splitData(b039Table, seed=42)

In [38]:
class SegSet(Dataset):
    """
    Segmentation dataset for grayscale or fluorescent image-mask pairs.
    """

    def __init__(self, dataTable, size=(256, 256), mode="auto"):
        self.dataTable = dataTable.reset_index(drop=True).copy()
        self.size = size
        self.mode = mode

    def __len__(self):
        return len(self.dataTable)

    def normalizeImage(self, imageArray, sourceName):
        selectedMode = self.mode

        if selectedMode == "auto":
            if "039" in sourceName or "FLUO" in sourceName.upper():
                selectedMode = "fluo"
            else:
                selectedMode = "gray"

        if selectedMode == "gray":
            return normalizeGray(imageArray)

        if selectedMode == "fluo":
            return normalizeFluorescent(imageArray)

        raise ValueError(f"Unsupported mode: {self.mode}")

    def __getitem__(self, index):
        row = self.dataTable.iloc[index]

        imageArray = readImage(row["image_path"])
        maskArray = readImage(row["mask_path"])

        imageArray = toTwoDimensional(imageArray).astype(np.float32)
        maskArray = toTwoDimensional(maskArray).astype(np.float32)

        imageArray = self.normalizeImage(imageArray, row["source"])
        maskArray = (maskArray > 0).astype(np.float32)

        imageArray = resizeImage(imageArray, self.size)
        maskArray = resizeMask(maskArray, self.size)

        imageArray = np.expand_dims(imageArray, axis=0)
        maskArray = np.expand_dims(maskArray, axis=0)

        splitName = row["split"] if "split" in row.index else "na"

        return {
            "image": torch.tensor(imageArray, dtype=torch.float32),
            "mask": torch.tensor(maskArray, dtype=torch.float32),
            "image_id": row["image_id"],
            "source": row["source"],
            "split": splitName,
            "dataset_index": index
        }

In [39]:
# create data table
b039Table = pairFiles(b039ImageDirectory, b039MaskDirectory, "BBBC039")

# split into train / validation / test tables
trainTable, validationTable, testTable = splitData(b039Table, seed=42)

# create dataset objects
trainingSet = SegSet(trainTable, size=(256, 256), mode="fluo")
validationSet = SegSet(validationTable, size=(256, 256), mode="fluo")
testSet = SegSet(testTable, size=(256, 256), mode="fluo")

print("BBBC039")
print("training size:", len(trainingSet))
print("validation size:", len(validationSet))
print("test size:", len(testSet))

BBBC039
training size: 139
validation size: 30
test size: 31


In [40]:
b006Table = pairB006(
    b006ImageDirectory,
    b006MaskDirectory,
    sourceName="BBBC006",
    channelTag="_w1"
)

trainTable, validationTable, testTable = splitData(b006Table, seed=42)

trainingSet = SegSet(trainTable, size=(256, 256), mode="gray")
validationSet = SegSet(validationTable, size=(256, 256), mode="gray")
testSet = SegSet(testTable, size=(256, 256), mode="gray")

print("training size:", len(trainingSet))
print("validation size:", len(validationSet))
print("test size:", len(testSet))

training size: 537
validation size: 115
test size: 116


In [41]:
# Assuming `fullDataset` is already defined and contains the complete dataset.
trainingLoader = makeLoader(trainingSet, batchSize=4, shuffle=True)
validationLoader = makeLoader(validationSet, batchSize=4, shuffle=False)

testNetwork = UNet(inputChannels=1, outputChannels=1, dropoutRate=0.3).to(DEVICE)

trainedNetwork, historyTable = trainNetwork(
    network=testNetwork,
    trainingLoader=trainingLoader,
    validationLoader=validationLoader,
    device=DEVICE,
    numEpochs=2,
    learningRate=1e-3,
    savingPath=None
)

print(historyTable.head())

NameError: name 'readImage' is not defined

In [ ]:
#plotting the training and validation loss, and validation Dice and IoU scores across epochs to visualize the training progress and performance of the model.

plt.figure(figsize=(8, 4))
plt.plot(historyTable["epoch"], historyTable["trainLoss"], label="Train Loss")
plt.plot(historyTable["epoch"], historyTable["validationLoss"], label="Validation Loss")
plt.xlabel("Epoch")
plt.ylabel("Loss")
plt.title("Training and Validation Loss")
plt.legend()
plt.grid(True)
plt.tight_layout()
plt.show()

plt.figure(figsize=(8, 4))
plt.plot(historyTable["epoch"], historyTable["validationDice"], label="Validation Dice")
plt.plot(historyTable["epoch"], historyTable["validationIoU"], label="Validation IoU")
plt.xlabel("Epoch")
plt.ylabel("Score")
plt.title("Validation Dice and IoU")
plt.legend()
plt.grid(True)
plt.tight_layout()
plt.show()